In [12]:

import easyocr
import cv2


In [61]:
# Install EasyOCR
# !pip install easyocr

import easyocr
import cv2
import re
import numpy as np

# Initialize reader
reader = easyocr.Reader(['en', 'de'])

# Load image - handles both file path and bytes
def load_image(image_input):
    if isinstance(image_input, str):
        image = cv2.imread(image_input)
    elif isinstance(image_input, bytes):
        nparr = np.frombuffer(image_input, np.uint8)
        image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    else:
        raise ValueError("Input must be file path (str) or image bytes")
    return image

def get_image_text(image_input):
    # Load image
    image = load_image(image_input)
    
    # Read text from original image
    results_original = reader.readtext(image)
    
    # Read rotated image for vertical text
    rotated = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    results_rotated = reader.readtext(rotated)
    
    # Combine all results with confidence > 70%
    all_text = []
    for (bbox, text, confidence) in results_original + results_rotated:
        if confidence >= 0.70:
            all_text.append(text)
    
    # Join all text
    combined_text = ' '.join(all_text)
    
    # Keep only important words
    words = combined_text.split()
    english_keywords = ['Sika', 'Primer', 'Aktivator', 'ml', 'Made', 'Switzerland', 
                        'Best', 'before', 'end', 'of', 'Batch', 'No', 'in', 'x']
    
    filtered_words = []
    for word in words:
        if (re.search(r'\d', word) or 
            any(keyword.lower() in word.lower() for keyword in english_keywords) or
            re.match(r'^[:\-/®@\.]+$', word)):
            filtered_words.append(word)
    
    cleaned_text = ' '.join(filtered_words)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def extract_product_info(text):
    result = {}
    
    # Clean text
    text = re.sub(r'\s+', ' ', text)
    
    # Extract Name (Sika Primer-XXX or Sika Aktivator-XXX)
    name_match = re.search(r'Sika[®@]?\s*((?:Primer|Aktivator)[\s\-]*\d+)', text, re.IGNORECASE)
    if name_match:
        result['Name'] = f"Sika {name_match.group(1).replace(' ', '-')}"
    
    # Extract ML (handles: 30ml, 250ml, 24x30ml, 24 x 30ml, etc.)
    ml_match = re.search(r'(\d+\s*x\s*\d+\s*ml|\d+\s*ml)', text, re.IGNORECASE)
    if ml_match:
        result['ML'] = ml_match.group(1).replace(' ', '')
    
    # Extract Best Before (MM/YYYY)
    best_before = re.search(r'(?:Best\s+before|before\s+end\s+of)[\s:]*(\d{2}/\d{4})', text, re.IGNORECASE)
    if best_before:
        result['Best before'] = best_before.group(1)
    
    # Extract Batch No (10 digits)
    batch_match = re.search(r'Batch[\s\-]*(?:No\.?)?[\s:\-]*(\d{10})', text, re.IGNORECASE)
    if batch_match:
        batch_number = batch_match.group(1)
        batch_pos = text.find(batch_number)
        
        # Extract Artikel (4-6 digits AFTER batch number)
        if batch_pos != -1:
            text_after_batch = text[batch_pos + len(batch_number):]
            artikel_match = re.search(r'\b(\d{4,6})\b', text_after_batch)
            if artikel_match:
                result['artikel'] = artikel_match.group(1)
    
    return result

# Main function
def extract_from_image(image_input):
    combined_text = get_image_text(image_input)
    print(combined_text)
    product_info = extract_product_info(combined_text)
    print(product_info)
    return product_info

# Usage:
# result = extract_from_image('your_image.jpg')  # For file path
# result = extract_from_image(image_bytes)        # For bytes
# print(result)

In [62]:
import fitz  # PyMuPDF
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import base64
import string
import pandas as pd
import io
import os


doc = fitz.open("P2111_new.pdf")
#doc = fitz.open("Primer269-1.pdf")


def make_image_lists(doc):
    ref = {}
    multi_images = []
    images_list = []
    
    print("Total Pages", len(doc))
    for page_number in range(len(doc)):

        page = doc[page_number]
        text = page.get_text()
        text_data = text.split("\n")

        charge =[val.strip().lower() for val in text_data if len(val) > 1]
        images = page.get_images(full=True) 
        

        if(len(charge)<1):
            if (page_number == len(doc)-1):
                multi_images.append(images_list)
                images_list = []   
            continue

        second_val = ""
        first_val = ""
        if("flaschennummer" in charge[-1]):
            second_val = charge[-1].split(" ")
            if(len(second_val) >=1):
                second_val = second_val[1]
            else:
                second_val = charge[charge.index("flaschennummer")+1]

        if("halb-charge" in charge):
            first_val = charge[charge.index("halb-charge")+1]



        text = ""
        artikle = ""

        for i, tex in enumerate(text_data):
            if(tex.startswith("PO")):
                text=tex

            if(tex.startswith("FERT-Artikel-Nummer")):
                artikle = text_data[i+1]


        images = page.get_images(full=True) 
        text_data = text.split(":")

        text = text.split('  ')[0].strip()
        if(len(text) == 0):
            print(len(text), text_data, page_number)
            continue

        images = page.get_images(full=True) 

        images_sorted = sorted(images, key=lambda x: x[0])
        #print(images_sorted)
        if(len(images_sorted)<4):
            #print(f"{page_number} rejected: {text.split(':')[1].strip()}, {len(doc)}")
            continue
        #else:
        #    print(f"{page_number} accepted: {text.split(':')[1].strip()}, {len(doc)}")

        for img_index, img in enumerate(images_sorted):

            xref = img[0]  
            if(img[3] <150):
                #print(img_index, "Small Image")
                continue
                
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            ext = base_image["ext"]  # e.g. 'png', 'jpeg'

            # Construct file name
            filename = f"{text}_{page_number}_{img_index}.{ext}"
            filename = filename.replace(" ", "_")  # avoid spaces
            print(filename)

            base_image = doc.extract_image(xref)  
            image_bytes = base_image["image"]  
            
            
            #visualize_image(image_bytes, page_number, img_index)
            #result=read_text(file_path=None, image_bytes=image_bytes)
            
            product_info = extract_from_image(image_bytes)
            
        if(page_number == 10):
            break
make_image_lists(doc)

Total Pages 35
PO:_19403497_0_0.jpeg
Sika IN 25Oml of: 11/2026 lontano Indossare Continuare Best before In NON Batch-No : 3010880729 1431 professionale! Made in in UN1866 Resin 489-0254778111 infiammabil ingestione end parecchiminuti 1-20068 Einaudi 1 8 1 1 611237 8 1 1 8 8 1 5 8 8 1 1 1 1 1 1 1 1 1 1
{'artikel': '1431'}
PO:_19403497_0_1.jpeg
Resin Sika Einaudi 120068 +39-0254778111 Aktivator-10O professionalei Best before end of: 11/2026 Batch-No.: 3010880729 1431 Made in Switzerland Einaudi +39 0254778111 UN1866 Resin originali 5 @ 1 1 1 5
{'Best before': '11/2026', 'artikel': '1431'}
PO:_19403497_0_2.jpeg
51395 6561 5
{}
PO:_19403497_0_3.jpeg

{}
PO:_19403601_1_0.jpeg
fikaSika Best 1431 909D Made in Switzerland UN1866 Resin Sika 5-7 '+36-1-3712020 25Oml [0880733 1 611237 098673 1 8 1 1 5 1 @ 1 1 5 1 1 4
{}
PO:_19403601_1_1.jpeg
Best before end of: 11/2026 Batch-No.: 3010880733 1431 Made in Switzerland Sika 5.7 H-2051 +36-1-3712020 UN1866 Resin 9 5 1 5'
{'Best before': '11/2026', 'ar